# Лабораторная работа № 1. Линейные методы

### Набор данных

- Выберите любой набор данных для задачи бинарной классификации. Если он был не для бинарной классификации, то объедините несколько классов.
- Преобразуйте его в числовой вид и нормализуйте.
- Разбейте его на тренировочную и тестовую часть.
- Выберите целевую функцию ошибки или качества для задачи бинарной классификации.
- Если после преобразования в числа в наборе данных слишком много признаков, выберите часть признаков из набора данных любым способом. Например, при помощи абсолютной корреляции с целевым (с заменой на ±1).

### Алгоритмы

- Реализуйте алгоритм линейной регрессии в матричном виде с гребневой регуляризацией. Можно использовать библиотеки для работы с матрицами. Преобразуйте его в алгоритм линейной классификации через замену целевого признака на ±1.
- Реализуйте алгоритм линейной классификации на основе градиентного спуска.
- Алгоритм должен поддерживать не менее трёх эмпирических рисков, Elastic Net регуляризацию и настраиваемую скорость градиентного спуска. Эмпирические риски должны быть именно для задачи линейной классификации, вычисляться через отступ. Производные и градиент должны быть вычислены аналитически.

### Задача

- Постарайтесь подобрать хорошие гиперпараметры для каждого алгоритма.
- Постройте кривую обучения со сглаженным эмпирическим риском на тренировочном множестве для линейной классификации.
- Постройте кривую обучения с целевой функцией ошибки или качества на тестовом множестве для линейной классификации. Не обязательно замерять целевую функцию на каждой итерации, если их много. Переберите разные разбиения на тренировочную и тестовую часть как при перекрёстной проверке. Отметьте доверительный интервал на графике.
- Отметьте на предыдущем графике значение целевой функции на тестовом множестве для линейной регрессии. Например, горизонтальной прямой.
- Постройте график зависимости значений коэффициентов линейной модели от значения коэффициента L1-регуляризации (коэффициент для L2 должен быть равен нулю). Постройте второй график для зависимости от коэффициента L2-регуляризации (для L1 коэффициент равен нулю). Желательно перебирать коэффициенты регуляризации в логарифмированном пространстве.

## Подготовка данных

Воспользуемся датасетом [Selfie2Anime](https://huggingface.co/datasets/huggan/selfie2anime), в нём 3500 фотографий женских лиц с меткой $-1$ и 3500 изображений женских аниме-персонажей с меткой $+1$. Исходные разбиения датасета объединил и сделал собственное разбиение 80/20 (обучающая/тестовая).

При первом запуске исходные Parquet-файлы скачиваются в `dataset/original`; при следующих запусках повторное скачивание не требуется. Каждое изображение переводится в RGB, уменьшается с 256×256 до 32×32, перегоняется в вектор из $32\cdot32\cdot3=3072$ признаков, все значения лежат в отрезке от 0 до 255, поэтому отнормируем поделив на 255.

Признаков слишком много (не охото долго ждать вычисления), поэтому давайте уменьшим их кол-во до 333 методом абсолютной корреляции с целевым (чем больше модуль, тем более явно признак разграничивает две категории объектов).

В `dataset/after_normalized` сохраняются готовые `train.npz` и `test.npz`, общая числовая выборка для будущей перекрёстной проверки.


## Целевая метрика качества

Для оценки и сравнения моделей выбрана **accuracy** — кол-во цспешных предсказаний делим на размер тестовой выборки.

Линейная модель вычисляет оценку $a(x)=w^Tx+b$, и мы считаем классом $+1$ (аниме), если $a(x)\geq0$, и классом $-1$ (реальный человек), если $a(x)<0$. Стремимся увеличить accuracy по максимуму.



In [1]:
from pathlib import Path

from utils.download import download_dataset
from utils.normalization import prepare_dataset

DATASET_DIR = Path("dataset")
REPOSITORY_ID = "huggan/selfie2anime"
FILENAMES = [
    "data/train-00000-of-00001.parquet",
    "data/test-00000-of-00001.parquet",
]
IMAGE_SIZE = 32
TEST_SIZE = 0.2
RANDOM = 24422442
FEATURES_COUNT = 333

parquet_paths = download_dataset(
    REPOSITORY_ID,
    FILENAMES,
    DATASET_DIR / "original",
)

dataset_info = prepare_dataset(
    parquet_paths,
    DATASET_DIR / "after_normalized",
    IMAGE_SIZE,
    TEST_SIZE,
    RANDOM,
    FEATURES_COUNT,
)
dataset_info


Датасет уже скачан.


{'all_samples': 7000,
 'train_shape': (5600, 333),
 'test_shape': (1400, 333),
 'classes': {-1: 'фотография', 1: 'аниме'}}

Закончили с подготовкой, переходим к следующему этапу!

## Алгоритмы

### Линейная регрессия с гребневой регуляризацией

Добавим первым столбцом матрицы признаков единицы. Полученную матрицу обозначим $X_b=[\mathbf 1\;X]$. Параметры объединим в вектор $\theta=(b,w)$. Целевая функция $Q(\theta)=\lVert X_b\theta-y\rVert_2^2+\alpha\lVert w\rVert_2^2$.

Матричное решение имеет вид $\theta=(X_b^T X_b+\alpha R)^{-1}X_b^T y.$

Модель возвращает оценку $a(x)=w^Tx+b$, а класс определяется по её знаку:

$$\widehat y=\begin{cases}+1,&a(x)\geq0,\\-1,&a(x)<0.\end{cases}$$


### Линейный классификатор с градиентным спуском

Начинаем с $\theta=(b,w)=0$ и проводим несколько итераций градиентного спуска.

Ошибка объекта вычисляется через отступ $M_i=y_i(w^Tx_i+b).$ Он удобен тем, что для обоих классов улучшает ответ, если увеличиваем отступ.

#### Поддерживаются три риска с аналитическими производными:

| Риск | $L(M)$ | $L'(M)$ |
|---|---|---|
| Логистический | $\ln(1+e^{-M})$ | $-\dfrac{1}{1+e^M}$ |
| Экспоненциальный | $e^{-M}$ | $-e^{-M}$ |
| Квадратичный hinge | $\max(0,1-M)^2$ | $-2(1-M)$ при $M<1$, иначе $0$ |

Функции реализованы в utils/loss_functions.py. 

Целевая функция с Elastic Net: $Q(w,b)=\frac1n\sum_{i=1}^{n}L(M_i)+\lambda_1\lVert w\rVert_1+\frac{\lambda_2}{2}\lVert w\rVert_2^2.$

Поскольку $b$ и $w$ объединены в $\theta=(b,w)$, градиент для всех параметров вычисляется одной формулой:

$$\nabla_\theta Q=\frac1nX_b^T\bigl(L'(M) * y\bigr)+\lambda_1\operatorname{sign}(\theta_{reg})+\lambda_2\theta_{reg},$$

где $\theta_{reg}=(0,w)$, чтобы не регуляризовать $b$. Первый элемент полученного градиента относится к $b$, остальные — к весам $w$.

Тип риска, количество итераций и шаг итераций градиентного спуска, $\lambda_1$, $\lambda_2$ настраиваются. algorithms/with_history.py сохраняет риск и параметры для будущих графиков.


In [2]:
from algorithms.gradient_classifier import GradientLinearClassifier
from algorithms.ridge_classifier import RidgeRegressionClassifier

Закончили со всеми приготовлениями и начинаем проводить эксперименты:

## Задача


In [3]:
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path

from experements.ridge_search import run_ridge_search
from experements.common import load_dataset
from experements.gradient_search import run_elastic_net_search, run_loss_search
from experements.l1 import run_l1
from experements.l2 import run_l2
from experements.train_risk_with_interval import collect_learning_curve_data
from experements.train_risk import run_train_risk
from utils.plots import (
    plot_coefficient_path,
    plot_gradient_search,
    plot_ridge_search,
    plot_test_accuracy,
    plot_train_risk,
)

IMAGE_DIR = Path("images")

X_train, y_train = load_dataset("dataset/after_normalized/train.npz")
X_test, y_test = load_dataset("dataset/after_normalized/test.npz")
print("datasets loaded")

datasets loaded


### Подбор гиперпараметра Ridge-классификатора

Переберём коэффициент $\alpha$ в логарифмической шкале. Каждую модель обучим на фиксированной тренировочной выборке и выберем значение с наибольшей accuracy. 


In [4]:
alphas = np.logspace(-5, 5, 100)
results = run_ridge_search(X_train, y_train, X_test, y_test, alphas)

print(f"Лучшее alpha: {results['best_alpha']:.4g}")
print(f"Test accuracy: {results['best_score']:.4f}")

figure = plot_ridge_search(results, IMAGE_DIR / "ridge_search.png")
plt.close(figure)


Лучшее alpha: 0.7055
Test accuracy: 0.9614


![Подбор коэффициента Ridge-регуляризации](images/ridge_search.png)


#### Вывод
Accuracy остаётся около $0.96$ при малых и относительно небольших $\alpha$. Без сильной регуляризации модель уже хорошо разделяет два класса (хорошо выбрал признаки). При $\alpha$ порядка девятисот-тысячи видим спад количества корректных предсказаний, видимо +- при этих значениях норма параметров становится ошутимой относительно нормы ошибки.


### Подбор гиперпараметров градиентного классификатора

Сначала сравним три вида риска в зависимости от размера шага градиентного спуска. Затем подберём коэффициенты Elastic Net. Обучаем на одних и тех же тренировочных данных и проверяем на одинаковых тестовых. Также уменьшено кол-во итераций градиента, потому что этот эксперимент и так дольше остальных вместе взятых...


In [5]:
losses = ["logistic", "exponential", "squared_hinge"]
learning_rates = np.logspace(-3, np.log10(0.05), 15)
l1_values = np.concatenate(([0.0], np.logspace(-2, np.log10(0.3), 5)))
l2_values = np.concatenate(([0.0], np.logspace(-1, 1, 5)))
MAX_ITER = 150


loss_results = run_loss_search(
    X_train,
    y_train,
    X_test,
    y_test,
    losses,
    learning_rates,
    MAX_ITER,
    10,
)

elastic_net_results = run_elastic_net_search(
    X_train,
    y_train,
    X_test,
    y_test,
    loss_results["best_loss"],
    loss_results["best_learning_rate"],
    l1_values,
    l2_values,
    MAX_ITER,
    10,
)

gradient_results = {**loss_results, **elastic_net_results}

for loss, scores in zip(
    gradient_results["losses"], gradient_results["loss_scores"]
):
    best_index = np.argmax(scores)
    print(
        f"{loss}: learning rate = {learning_rates[best_index]:.4g}, "
        f"test accuracy = {scores[best_index]:.4f}"
    )

print(
    f"Лучший риск: {gradient_results['best_loss']}, "
    f"accuracy: {gradient_results['best_loss_score']:.4f}"
)
print(
    f"Лучшая скорость: {gradient_results['best_learning_rate']:.4g}, "
    f"accuracy: {gradient_results['best_loss_score']:.4f}"
)
print(
    f"Лучший L1: {gradient_results['best_l1']:.4g}, "
    f"accuracy: {gradient_results['best_score']:.4f}"
)
print(
    f"Лучший L2: {gradient_results['best_l2']:.4g}, "
    f"accuracy: {gradient_results['best_score']:.4f}"
)

figure = plot_gradient_search(
    gradient_results, IMAGE_DIR / "gradient_search.png"
)
plt.close(figure)


logistic: learning rate = 0.05, test accuracy = 0.9536
exponential: learning rate = 0.01635, test accuracy = 0.9557
squared_hinge: learning rate = 0.05, test accuracy = 0.9579
Лучший риск: squared_hinge, accuracy: 0.9579
Лучшая скорость: 0.05, accuracy: 0.9579
Лучший L1: 0, accuracy: 0.9579
Лучший L2: 0, accuracy: 0.9579


![Подбор гиперпараметров градиентного классификатора](images/gradient_search.png)


#### Вывод
Лучший результат показал squared hinge: при $\eta=0.05$ test accuracy равна $0.9579$. Экспоненциальный риск дал $0.9557$, а логистический — $0.9536$. Экспоненциональное уже начило переполняться, поэтому была сделана отсечка (спроецировал в непереполняемый интервал), поэтому не стал больше увеличивать максимальный шаг.

Elastic Net не улучшил результат: максимальная accuracy $0.9579$ получена при $\lambda_1=\lambda_2=0$. Значит, после предварительного отбора 333 признаков дополнительная регуляризация для выбранной модели не понадобилась (модель не переобучилась).

Результат довольно близок к полученному от ridge-классификатора. 

### Кривая эмпирического риска

Обучим выбранную модель на всей тренировочной выборке в течение 100 итераций. Для сглаживания будем брать среднее за последние 10 итераций.


In [6]:
train_risk_history = run_train_risk(
    X_train, y_train, gradient_results, 100
)
figure = plot_train_risk(
    train_risk_history["iteration"],
    train_risk_history["risk"],
    10,
    IMAGE_DIR / "train_risk.png",
)
plt.close(figure)


![Сглаженная кривая эмпирического риска](images/train_risk.png)


#### Вывод
 Эмпирический риск монотонно уменьшается. На первых итерациях есть скачки, но начиная с 10 итерации мы +- у оптимума и он практически монотонно идёт. Я уменьшил кол-во итераций с 1000 до 100, потому что на том вообще скучно было.


### Кривая обучения с доверительным интервалом

Сделаем 50 выборок в соотношении 80/20. На графике покажем среднюю accuracy градиентной модели, 90% доверительный интервал и среднее качество Ridge на тех же разбиениях.

Для каждой сохранённой итерации градиентного спуска получим 50 значений accuracy по ним получим среднее, а после заюзаем в формулке распределения Стьюдента.


In [7]:
X_all, y_all = load_dataset("dataset/after_normalized/all_features.npz")
learning_curve_results = collect_learning_curve_data(
    X_all,
    y_all,
    gradient_results,
    results["best_alpha"],
    FEATURES_COUNT,
    runs_count=50,
    test_size=0.2,
    random_state=RANDOM,
    max_iter=50,
    history_step=10,
    confidence=0.99,
)
print(
    "Gradient final accuracy:",
    f"{learning_curve_results['gradient_mean'][-1]:.4f}",
)
print("Ridge accuracy:", f"{learning_curve_results['ridge_mean']:.4f}")
print(
    f"Gradient {learning_curve_results['confidence']:.0%} confidence interval:",
    f"[{learning_curve_results['gradient_lower'][-1]:.4f}, "
    f"{learning_curve_results['gradient_upper'][-1]:.4f}]",
)
print(
    f"Ridge {learning_curve_results['confidence']:.0%} confidence interval:",
    f"[{learning_curve_results['ridge_lower']:.4f}, "
    f"{learning_curve_results['ridge_upper']:.4f}]",
)

figure = plot_test_accuracy(
    learning_curve_results, IMAGE_DIR / "learning_curve.png"
)
plt.close(figure)


Gradient final accuracy: 0.9507
Ridge accuracy: 0.9574
Gradient 99% confidence interval: [0.9480, 0.9534]
Ridge 99% confidence interval: [0.9557, 0.9592]


![Кривая обучения с доверительным интервалом](images/learning_curve.png)


#### Вывод
 Средняя test accuracy градиентного классификатора быстро растёт на первых итерациях и затем стабилизируется около $0.95$. По 50 случайным разбиениям её 99%-й доверительный интервал равен $[0.9480, 0.9534]$. Ridge достигает средней accuracy $0.9574$ с интервалом [0.9557, 0.9592], поэтому можем сделать вывод что с вероятностью в 99% градиентный классификатор хуже, чем ridge-классификатор.

 Даже на 99% интервал почти не видно...


### Влияние L1-регуляризации

Положим $\lambda_2=0$ и переберём $\lambda_1$ в логарифмическом пространстве. Всего модель содержит 333 коэффициента, поэтому возьмём случайно 10 коэффициентов.

In [8]:
l1_path_results = run_l1(
    X_train,
    y_train,
    gradient_results,
    np.logspace(-6, -1, 15),
    max_iter=MAX_ITER,
)
figure = plot_coefficient_path(
    l1_path_results, 10, IMAGE_DIR / "l1_coefficients.png", RANDOM
)
plt.close(figure)


![Зависимость коэффициентов от L1](images/l1_coefficients.png)


#### Вывод
При малых значениях L1 коэффициенты почти не меняются. Начиная примерно с $\lambda_1=10^{-3}$ - $10^{-4}$, их модули уменьшаются. При росте регуляризации все веса приближаются к 0, что вообщем то ожидаемо.


### Влияние L2-регуляризации

Положим $\lambda_1=0$ и переберём $\lambda_2$ в логарифмическом пространстве. Покажем те же 10 случайно выбранных коэффициентов, чтобы результат можно было сравнить с L1-регуляризацией.

In [9]:
l2_path_results = run_l2(
    X_train,
    y_train,
    gradient_results,
    np.logspace(-4, 0, 15),
    max_iter=MAX_ITER,
)
figure = plot_coefficient_path(
    l2_path_results, 10, IMAGE_DIR / "l2_coefficients.png", RANDOM
)
plt.close(figure)


![Зависимость коэффициентов от L2](images/l2_coefficients.png)


#### Вывод
 С ростом L2 модули большинства выбранных коэффициентов уменьшаются. Некоторые веса меняются немонотонно или пересекают ноль, поскольку модель может перераспределять влияние между коррелирующими признаками. Но в общем L2 меньше устремляет коэфы к нулю (потому что при коэфах меньших единиицы возведение в квадрат их уменьшает относительно L1 регуляризации).